In [1]:
#Importaciones

import os, math, random
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils


In [2]:
#Device info

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
LR = 2e-4
Z_DIM = 64          # tamaño del vector de ruido
EPOCHS = 2          # entrenamiento corto
SAMPLES_DIR = "samples"
os.makedirs(SAMPLES_DIR, exist_ok=True)
torch.manual_seed(42); random.seed(42)

In [3]:
# 1) Datos: MNIST 28x28 en [-1,1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
mnist = datasets.MNIST(root="data", train=True, download=True, transform=transform)
# Subconjunto pequeño para ir rápido (p. ej., 10k ejemplos)
idx = list(range(10000))
subset = Subset(mnist, idx)
loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# 2) Modelos (MLP) -----------------------------------------------------------
class Generator(nn.Module):
    def __init__(self, z_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 28*28),
            nn.Tanh()   # salida en [-1,1]
        )
    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

G = Generator(Z_DIM).to(DEVICE)
D = Discriminator().to(DEVICE)

# 3) Pérdida y optimizadores
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

def sample_and_save(step, n=16):
    with torch.no_grad():
        z = torch.randn(n, Z_DIM, device=DEVICE)
        fake = G(z).cpu()
        grid = utils.make_grid(fake, nrow=int(math.sqrt(n)), normalize=True, value_range=(-1,1))
        utils.save_image(grid, os.path.join(SAMPLES_DIR, f"step_{step:06d}.png"))

# 4) Entrenamiento -----------------------------------------------------------
step = 0
for epoch in range(EPOCHS):
    for real, _ in loader:
        real = real.to(DEVICE)
        bs = real.size(0)

        # ---- Entrenar Discriminador
        D.zero_grad()
        # etiquetas reales=1, falsas=0
        y_real = torch.ones(bs, 1, device=DEVICE)
        y_fake = torch.zeros(bs, 1, device=DEVICE)

        # pérdida con reales
        out_real = D(real)
        loss_D_real = criterion(out_real, y_real)

        # generar falsos
        z = torch.randn(bs, Z_DIM, device=DEVICE)
        fake = G(z).detach()
        out_fake = D(fake)
        loss_D_fake = criterion(out_fake, y_fake)

        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        opt_D.step()

        # ---- Entrenar Generador
        G.zero_grad()
        z = torch.randn(bs, Z_DIM, device=DEVICE)
        fake = G(z)
        # aquí queremos que D piense que los falsos son reales (etiqueta=1)
        out = D(fake)
        loss_G = criterion(out, y_real)
        loss_G.backward()
        opt_G.step()

        if step % 100 == 0:
            print(f"epoch {epoch} step {step} | loss_D={loss_D.item():.3f} loss_G={loss_G.item():.3f}")
            sample_and_save(step, n=16)
        step += 1

print("Entrenamiento finalizado. Revisa la carpeta 'samples/' para ver imágenes generadas.")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.28MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 603kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.85MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.94MB/s]


epoch 0 step 0 | loss_D=1.449 loss_G=0.666
epoch 0 step 100 | loss_D=1.219 loss_G=0.687
epoch 1 step 200 | loss_D=1.101 loss_G=1.012
epoch 1 step 300 | loss_D=0.865 loss_G=0.996
Entrenamiento finalizado. Revisa la carpeta 'samples/' para ver imágenes generadas.
